
# <center>**ClarIA: Claim Assistance Resolution**</center>

---

**Técnicas de Inteligencia Artificial - Deep Learning**

**Martina Ibarra | Elena Salomon | Florencia Nebot**

---


# Introducción y motivación

Las empresas reciben miles de quejas de consumidores diariamente
pero carecen de herramientas para identificar automáticamente los problemas. Actualmente, las quejas se analizan
manualmente o se clasifican sólo superficialmente por producto/categoría, sin
detectar las causas raíz que podrían prevenir futuras quejas si se corrigen. Esto genera desperdicio de recursos reaccionando a síntomas individuales en lugar de resolver problemas estructurales.

En este proyecto proponemos el desarrollo de un agente inteligente basado en Deep Learning que, mediante técnicas de Retrieval-Augmented Generation (RAG) y Ollama, permite consultar el dataset de quejas financieras en lenguaje natural. El sistema no solo recupera registros relevantes, sino que genera respuestas analíticas que ayudan a identificar:

1. Principales problemas por producto financiero

2. Tendencias emergentes

3. Áreas críticas que requieren atención estratégica

Este enfoque permite transformar un conjunto masivo de comentarios no estructurados en insights accionables para la toma de decisiones.


**Dataset**

https://www.kaggle.com/datasets/selener/consumer-complaint-database

**Metadata**

* **"Date received"**: La fecha en que la CFPB recibió la queja.
* **"Product"**: El tipo de producto que el consumidor identificó en la queja.
* **"Sub-product"**: El tipo de subproducto que el consumidor identificó en la queja.
* **"Issue"**: El problema que el consumidor identificó en la queja.
* **"Sub-issue"**: El subproblema que el consumidor identificó en la queja.
* **"Consumer complaint narrative"**: Descripción del consumidor sobre "lo que sucedió" en la queja.
* **"Company public response"**: La respuesta opcional y pública de la empresa a la queja del consumidor.
* **"Company"**: La empresa sobre la que trata la queja.
* **"State"**: El estado de la dirección de correo proporcionada por el consumidor.
* **"ZIP code"**: El código postal de envío proporcionado por el consumidor.
* **"Tags"**: Datos que facilitan la búsqueda y clasificación de las quejas.
* **"Consumer consent provided?"**: Identifica si el consumidor optó por publicar el relato de su queja.
* **"Submitted via"**: Cómo se envió la queja a la CFPB.
* **"Date sent to company"**: La fecha en que la CFPB envió la queja a la empresa.
* **"Company response to consumer"**: Cómo respondió la empresa.
* **"Timely response?"**: Si la empresa dio una respuesta a tiempo.
* **"Consumer disputed?"**: Si el consumidor disputed la respuesta de la empresa.
* **"Complaint ID"**: El número de identificación único para una queja.

https://cfpb.github.io/api/ccdb/fields.html

In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn datasets transformers torch sentence-transformers umap-learn hdbscan faiss-cpu langchain langchain-community langchain-experimental ollama

In [ ]:
# Importamos librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import torch
import hdbscan
import umap
import time
import faiss
import subprocess
import spacy

from tqdm.auto import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding, DistilBertTokenizer, DistilBertForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

from langchain_community.llms import Ollama
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [ ]:
# Instalamos mas librerias.
#!pip install pandas matplotlib seaborn langchain langchain-text-splitters

In [ ]:
# Quitar el límite del ancho de las columnas para leer comentarios
pd.set_option('display.max_colwidth', None)


In [ ]:
# Nos conectamos a nuestra cuenta de Google Drive para acceder a los datos.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/MIT/ClarIA/rows.csv'
#path = '/content/drive/MyDrive/Diploma UTEC/Deep Learning/ClarIA/rows.csv'
df = pd.read_csv(path)

# Tratamiento de datos y análisis exploratorio

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
#Eliminamos columnas que no son relevantes para nuestro proyecto
df = df.drop(columns=['State', 'ZIP code', 'Tags', 'Consumer consent provided?','Company public response'])

In [ ]:
df.info()

**Observaciones**

Este dataset de quejas de la CFPB cuenta con 1.282.355 registros, donde campos clave como Product, Issue y Company están totalmente completos. Sin embargo, dado que el objetivo es desarrollar un sistema de resolucion de quejas, la columna crítica es Consumer complaint narrative, ya que contiene el texto semántico que el modelo de lenguaje necesita para "leer" y responder consultas. Como solo el 30% de las quejas (383.564 registros) incluyen este relato, limpiaremos el dataset para quedarnos únicamente con las filas donde esta columna está completa, asegurando que nuestro buscador de FAISS e inteligencia artificial trabaje solo con datos que aporten contexto real y no con entradas vacías.

## Nulos y Duplicados

In [ ]:
# 1. Definir la columna clave
col_comentarios = 'Consumer complaint narrative'

# 2. Contar nulos
nulos = df[col_comentarios].isnull().sum()
porcentaje_nulos = (nulos / len(df)) * 100

# 3. Contar duplicados (solo en la columna de texto, ignorando nulos)
# Filtramos los nulos primero para que no cuenten como duplicados entre sí
comentarios_validos = df[df[col_comentarios].notnull()][col_comentarios]
duplicados = comentarios_validos.duplicated().sum()
porcentaje_duplicados = (duplicados / len(comentarios_validos)) * 100

# 4. Mostrar resultados formateados
print("=== Auditoría del Dataset ===")
print(f"Total de registros: {len(df):,}")
print(f"Comentarios Nulos:  {nulos:,} ({porcentaje_nulos:.2f}%)")
print(f"Comentarios Duplicados: {duplicados:,} ({porcentaje_duplicados:.2f}%)")

# 5. Visualizar un par de ejemplos duplicados (para entender por qué se repiten)
if duplicados > 0:
    print("\n--- Ejemplo de comentarios que se repiten ---")
    print(comentarios_validos[comentarios_validos.duplicated()].head(2).values)

**Observaciones**
Se encontraron nulos y duplicados, procederemos a eliminar estos comentarios para no perder capacidad RAM procesando información repetida, ni tampoco generar errores de ejecución por datos vacíos.



In [ ]:
#Nos quedamos unicamente con las filas que tienen el claim escrito y eliminamos
df_filtered = df[df['Consumer complaint narrative'].notna()]
df_clean = df_filtered.drop_duplicates(subset=['Consumer complaint narrative'])

## Selección de datos

Revisamos las empresas de las cuales tenemos reclamos.

Como tenemos diversidad de tipos de empresas, le pasamos el listado a Gemini para que nos ayude a clasificar las empresas según su tipo. De esta forma podremos quedarnos unicamente con los bancos.

Consideramos que la naturaleza de los reclamos y la normativa aplicable difieren significativamente según el tipo de institución financiera (ej. Bancos vs. Bureau de crédito). Al centrarnos solo en bancos, garantizamos que el análisis de los reclamos sea consistente y comparable.

In [ ]:
def clasificar_entidad(nombre):
    nombre = str(nombre).upper()

    # 1. OTROS (Filtramos primero lo que NO es servicio directo o es B2B)
    otros_keywords = [
        'LLP', 'LLC', 'PC', 'PLLC', 'ATTORNEY', 'LAW', 'LEGAL', 'RECOVERY',
        'COLLECTION', 'ADJUSTMENT', 'ARBITRATION', 'BUREAU', 'EQUIFAX',
        'EXPERIAN', 'TRANSUNION', 'ACRANET', 'SERVICES', 'MANAGEMENT'
    ]
    if any(k in nombre for k in otros_keywords):
        return "Otros"

    # 2. FINTECH & E-BANKING (Marcas digitales específicas)
    fintech_ebanking = {
        'COINBASE': 'Fintech', 'PAYPAL': 'Fintech', 'XOOM': 'Fintech',
        'BITCOIN': 'Fintech', 'XAPO': 'Fintech', 'WINKLEVOSS': 'Fintech',
        'ALLY': 'e-banking', 'SCHWAB': 'e-banking', 'CHASE': 'Banco', # Chase es tradicional
        'NETSPEND': 'Fintech', 'SQUARE': 'Fintech', 'VENMO': 'Fintech'
    }
    for key, val in fintech_ebanking.items():
        if key in nombre:
            return val

    # 3. BANCOS (Palabras clave de banca tradicional)
    banco_keywords = ['BANK', 'BANCORP', 'BANCORPORATION', 'NATIONAL ASSOCIATION', 'N.A.', 'TRUST']
    if any(k in nombre for k in banco_keywords):
        return "Banco"

    # 4. PRESTADORAS DE CRÉDITO (Consumo, Hipotecas y Autos)
    prestadora_keywords = [
        'MORTGAGE', 'LENDING', 'LOAN', 'FINANCE', 'FINANCIAL', 'CREDIT',
        'ACCEPTANCE', 'FUNDING', 'CASH', 'MONEY', 'HONDA', 'FORD', 'NISSAN',
        'BMW', 'TOYOTA', 'HYUNDAI', 'CHRYSLER'
    ]
    if any(k in nombre for k in prestadora_keywords):
        return "Prestadora de crédito"

    # 5. DEFAULT (Si no entra en ninguna, lo mandamos a otros para revisión)
    return "Otros"

# --- EJECUCIÓN ---

df_clean['Categoria_Negocio'] = df_clean['Company'].apply(clasificar_entidad)

print("Distribución de la clasificación:")
print(df_clean['Categoria_Negocio'].value_counts())

# Ver una muestra de la clasificación
display(df_clean[['Company', 'Categoria_Negocio']].drop_duplicates().head(20))

In [ ]:
# Agrupamos por la categoría asignada y verificamos algunas compañias por tipo de empresa
ejemplos_por_tipo = df_clean.groupby('Categoria_Negocio')['Company'].unique().apply(lambda x: x[:10])

for categoria, empresas in ejemplos_por_tipo.items():
    print(f"=== TIPO DE NEGOCIO: {categoria} ===")
    for i, nombre in enumerate(empresas, 1):
        print(f"  {i}. {nombre}")
    print("-" * 30)

In [ ]:
df_clean = df_clean[df_clean['Categoria_Negocio'] == 'Banco'].copy()

In [ ]:
# Configuración de paleta de colores
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] # Una fuente limpia y legible
color_claria = '#ff4f63'

# 1. Preparación de la figura
# Usamos constrained_layout=True para que los tamaños sean idénticos y proporcionales
fig, axs = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
fig.suptitle('Análisis de Quejas',
             fontsize=20, color='#333333', alpha=0.8)

# --- Subplot 1: Distribución por Producto ---
data_prod = df_clean['Product'].value_counts().sort_values(ascending=True)
data_prod.plot(kind='barh', ax=axs[0, 0], color=color_claria, width=0.7)
axs[0, 0].set_title('Distribución por Producto', fontsize=14, color='#555555')
axs[0, 0].set_xlabel('Número de Reclamos', fontsize=10, alpha=0.7)
axs[0, 0].tick_params(labelsize=9)

# --- Subplot 2: Top 15 Motivos (Issue) ---
data_issue = df_clean['Issue'].value_counts().head(15).sort_values(ascending=True)
data_issue.plot(kind='barh', ax=axs[0, 1], color=color_claria, width=0.7)
axs[0, 1].set_title('Top 15 Motivos de Reclamos', fontsize=14, color='#555555')
axs[0, 1].set_xlabel('Número de Reclamos', fontsize=10, alpha=0.7)
axs[0, 1].tick_params(labelsize=9)

# --- Subplot 3: Top 10 Compañías ---
data_comp = df_clean['Company'].value_counts().head(10).sort_values(ascending=True)
data_comp.plot(kind='barh', ax=axs[1, 0], color=color_claria, width=0.7)
axs[1, 0].set_title('Top 10 Compañías con más Reclamos', fontsize=14, color='#555555')
axs[1, 0].set_xlabel('Número de Reclamos', fontsize=10, alpha=0.7)
axs[1, 0].tick_params(labelsize=9)

# --- Subplot 4: Tendencia Temporal ---
# Resampleamos por mes para la línea de tendencia
df_clean['Date received'] = pd.to_datetime(df_clean['Date received'])
data_time = df_clean.set_index('Date received').resample('M').size()
axs[1, 1].plot(data_time.index, data_time.values, color=color_claria, linewidth=1.5, alpha=0.8)
axs[1, 1].set_title('Evolución temporal de Reclamos', fontsize=14, color='#555555')
axs[1, 1].set_ylabel('Cantidad de Reclamos', fontsize=10, alpha=0.7)
axs[1, 1].grid(axis='y', linestyle='--', alpha=0.3)
axs[1, 1].tick_params(labelsize=9, rotation=60)

# Eliminamos bordes innecesarios (spines) para un look más moderno
for ax in axs.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')

plt.show()

**Observaciones**

Para este análisis, aplicaremos un filtro de segmentación para enfocarnos exclusivamente en la banca tradicional, conservando solo los registros clasificados como "Banco" y eliminando el resto de las categorías (Fintech, Prestadoras de crédito, etc.). El objetivo de esta depuración es eliminar el "ruido" que generan otros sectores financieros con modelos de negocio distintos.

Del análisis descriptivo de los datos de los registros de bancos podemos ver:
1. Una alta concentración de demanda en productos de Mortgage (Hipotecas) y Credit Cards (tarjetas de crédito), estos representan el mayor volumen de reclamos.

2. Los motivos principales ("Managing an account" y "Account opening") indican que las causas raíz no son solo fallas financieras, sino problemas en la experiencia de usuario y trámites burocráticos.

3. El volumen está fuertemente inclinado hacia tres entidades principales (Citibank, BoA, JPMorgan). Para el agente, esto implica que debe ser capaz de distinguir entre quejas generales del sistema y problemas específicos de la operativa de cada banco.

4. Inestabilidad Temporal: La tendencia muestra picos de actividad (especialmente hacia 2018) que sugieren eventos sistémicos. El agente automático permitiría detectar estos "picos" en tiempo real, pasando de una reacción tardía a una solución proactiva de problemas estructurales.

### **Tratamiento de datos**

Al trabajar con comentarios, sabemos que el lenguaje humano es ruidoso por defecto, para que nuestro modelo no aprenda sobre ese ruido, eliminaremos elementos que no aportan valor semántico pero que sí generan confusión en los modelos de inteligencia artificial.
1. Eliminar outliers
2. Estandarizar a minúsculas
3. Remover el ruido de la anonimización (las secuencias de "X", por datos confidenciales)
4. Limpiar caracteres especiales


In [ ]:
# 1. Calculamos la longitud de las narrativas en el dataset muestreado
df_clean['longitud'] = df_clean['Consumer complaint narrative'].str.len()

# 2. Calculamos los estadísticos principales (Cuartiles)
estadisticos = df_clean['longitud'].describe(percentiles=[.25, .50, .75, .90, .95])

# 3. Definimos outliers usando el método de Rango Intercuartílico (IQR)
Q1 = estadisticos['25%']
Q3 = estadisticos['75%']
IQR = Q3 - Q1
limite_superior_iqr = Q3 + 1.5 * IQR
limite_inferior_iqr = Q1 - 1.5 * IQR

# 4. Conteo de outliers
outliers_superiores = df_clean[df_clean['longitud'] > limite_superior_iqr]
outliers_inferiores = df_clean[df_clean['longitud'] < limite_inferior_iqr] # Basado en nuestro criterio de calidad

print("--- ANÁLISIS DE LONGITUD DE CARACTERES ---")
print(estadisticos)
print("\n--- DETECCIÓN DE OUTLIERS ---")
print(f"Límite superior estadístico (IQR): {int(limite_superior_iqr)} caracteres")
print(f"Cantidad de quejas 'eterno-largas' (> IQR): {len(outliers_superiores)}")
print(f"Cantidad de quejas 'muy cortas' (< IQR): {len(outliers_inferiores)}")

In [ ]:
# Filtramos un pequeño subconjunto que esté cerca de los 150 caracteres
ejemplos_frontera = df_clean[df_clean['Consumer complaint narrative'].str.len().between(140, 160)]

print(f"Se encontraron {len(ejemplos_frontera)} quejas en el rango de 140-160 caracteres.\n")

# Mostramos las primeras 3 para analizar su "riqueza" de información
for i, texto in enumerate(ejemplos_frontera['Consumer complaint narrative'].head(3), 1):
    print(f"Ejemplo {i} ({len(texto)} caracteres):")
    print(f"'{texto}'")
    print("-" * 50)

In [ ]:
MIN_CHARS = 100
MAX_CHARS = 1500

df_filtered = df_clean[
    (df_clean['Consumer complaint narrative'].str.len() >= MIN_CHARS) &
    (df_clean['Consumer complaint narrative'].str.len() <= MAX_CHARS)
].copy()

eliminados_cortos = len(df_clean[df_clean['Consumer complaint narrative'].str.len() < MIN_CHARS])
eliminados_largos = len(df_clean[df_clean['Consumer complaint narrative'].str.len() > MAX_CHARS])

print(f"--- REPORTE DE RECORTE ---")
print(f"✅ Quejas que pasaron el filtro: {len(df_filtered)}")
print(f"❌ Eliminadas por ser muy cortas (<{MIN_CHARS}): {eliminados_cortos}")
print(f"❌ Eliminadas por ser muy largas (>{MAX_CHARS}): {eliminados_largos}")

LIMITE_CASOS = 250

df_clean_dos = df_filtered.groupby('Product', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), LIMITE_CASOS), random_state=42)
)

In [ ]:
def clean_complaints(text):
    if not isinstance(text, str): return ""
    # 1. Minúsculas
    text = text.lower()
    # 2. Borrar secuencias de X (xxxx, xx/xx/xxxx, etc.)
    text = re.sub(r'[x]{2,}', '', text)
    # 3. Borrar caracteres especiales y números (opcional pero recomendado)
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. Limpiar espacios
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Creamos la columna limpia
df_clean_dos['clean_narrative'] = df_clean_dos['Consumer complaint narrative'].apply(clean_complaints)

In [ ]:
# Configuramos pandas para que no corte el texto en la visualización
pd.set_option('display.max_colwidth', None)

# Creamos una vista estética con estilo CSS
df_clean_dos[['Consumer complaint narrative', 'clean_narrative']].head(1).style.set_properties(**{
    'text-align': 'left',
    'white-space': 'normal',
    'border': '1px solid #ccc'
}).set_table_styles([
    dict(selector='th', props=[('background-color', '#ff4f63'), ('color', 'white'), ('font-weight', 'bold')])
])

###**Submuestreo controlado**
Como vimos en las salidas anteriores, tenemos disparidad en cantidad de quejas entre productos, para que el modelo no quede sesgado hacia esos productos realizaremos una recategorización de productos y un submuestreo controlado, limitaremos cada categoría de producto a un máximo de 250 casos



In [ ]:
# 1. Definimos un diccionario de productos para disminuir categorías de los mismos
mapa_categorias = {
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting',
    'Credit card or prepaid card': 'Cards',
    'Credit card': 'Cards',
    'Prepaid card': 'Cards',
    'Mortgage': 'Mortgages',
    'Checking or savings account': 'Banking Services',
    'Bank account or service': 'Banking Services',
    'Money transfer, virtual currency, or money service': 'Money Transfers',
    'Debt collection': 'Debt Collection',
    'Vehicle loan or lease': 'Loans',
    'Consumer Loan': 'Loans',
    'Student loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans'
}

df_clean_dos['Product_Consolidated'] = df_clean_dos['Product'].map(mapa_categorias).fillna('Other')


# 2. Aplicamos el Undersampling
LIMITE_CASOS = 250

df_sample = df_clean_dos.groupby('Product_Consolidated', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), LIMITE_CASOS), random_state=42)
)

df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print("Distribución consolidada y balanceada:")
print(df_sample['Product_Consolidated'].value_counts())

In [ ]:
df_sample['Product_Consolidated'].info()

**Observaciones**

Al usar lacategorización, groupby y sample, garantizas que los productos con miles de quejas no eclipsen a los más pequeños, creando un subconjunto equitativo que acelera el procesamiento sin perder la diversidad de temas.

De un dataset que contenía casi 1.3M de filas, logramos uno de 1941 filas sin nulos, sin duplicados y más balanceado.

# Validación de queja

Implementamos un pipeline de Preprocesamiento Basado en Agentes que actúa como un evaluador de calidad. Este componente analiza la estructura lógica de la queja y determina su utilidad para el sistema RAG. Al identificar y etiquetar el 'Ruido' (comentarios sin sustancia o incoherentes) antes de la indexación vectorial, garantizamos que la base de conocimientos de ClarIA mantenga una alta densidad de información relevante, minimizando así las alucinaciones del modelo en la etapa de respuesta.

In [ ]:
!pip install ollama

In [ ]:
# 1. Instalamos la dependencia zstd
!sudo apt-get install -y zstd

# 2. Instalamos Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciamos el servidor en segundo plano
subprocess.Popen(['ollama', 'serve'])
time.sleep(15)

# 4. Descargamos el modelo para la clasificación
!ollama pull llama3.2

In [ ]:
# Verificamos si el archivo existe antes de lanzarlo´
import os
if os.path.exists('/usr/local/bin/ollama'):
    print("✅ Binario de Ollama encontrado.")
    # Lanzamos el servidor usando la ruta completa
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'])
    print("⏳ Esperando 15 segundos a que el servidor inicie...")
    time.sleep(15)
else:
    print("❌ El binario no se encontró. Reintenta ejecutar la celda de instalación (paso 1).")

In [ ]:
import subprocess
import time
import ollama

def iniciar_ollama():
    try:
        # Intentamos una comunicación rápida para ver si ya está prendido
        ollama.list()
        print("✅ Ollama ya está ejecutándose.")
    except Exception:
        print("⚠️ Ollama no responde. Intentando encenderlo...")
        # Ejecuta 'ollama serve' en segundo plano
        # shell=True es necesario en Windows; en Mac/Linux podrías no necesitarlo
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Le damos unos segundos para que cargue antes de seguir
        time.sleep(5)
        print("🚀 Servidor Ollama iniciado.")

# Llamamos a la función
iniciar_ollama()

In [ ]:
from tqdm.auto import tqdm

# Activamos la barra de progreso para Pandas
tqdm.pandas()

print(f"Iniciando el procesamiento de {len(df_sample)} quejas.")

In [ ]:
import json
import re

def filtro_completo_ollama(texto):
    try:
        prompt = f"""
        Classify if the following financial complaint is ACTIONABLE or NOISE.
        ACTIONABLE: Specific issue to fix (errors, fraud, disputes).
        NOISE: General venting, insults, or incoherent text.

        Text: "{str(texto)[:500]}"

        Response format: JSON with keys 'classification' and 'reason'

        """

        response = ollama.generate(model='llama3.2', prompt=prompt)
        res_text = response['response'].strip()

        # 1. Intentar extraer el JSON usando Regex (por si el modelo añade texto antes o después)
        match = re.search(r'\{.*\}', res_text, re.DOTALL)
        if match:
            res_text = match.group(0)

        # 2. Limpiar posibles caracteres extraños
        res_text = res_text.replace('\n', ' ').strip()

        res_json = json.loads(res_text)
        return res_json.get('classification', 'ACTIONABLE'), res_json.get('reason', 'Validated')

    except Exception as e:
        # Si el JSON viene mal formado, lo marcamos para revisarlo luego
        return "ACTIONABLE", f"Error de formato: {str(e)}"

In [ ]:
import json

df_sample[['Filtro', 'Motivo_Filtro']] = df_sample['clean_narrative'].progress_apply(
    lambda x: pd.Series(filtro_completo_ollama(x))
)

# GUARDAMOS RESULTADOS
df_sample.to_csv('dataset_claria_clasificado.csv', index=False)

print("\n✅ ¡Procesamiento completado con éxito!")
print(f"Total de quejas procesadas: {len(df_sample)}")
print(f"Quejas descartadas como RUIDO: {len(df_sample[df_sample['Filtro'] == 'NOISE'])}")

### **Observaciones**

El análisis de sentimiento con Ollama 3.2 reveló que solo un 2.47% de los comentarios (48 de 1941) carecían de valor o presentaban contenido incoherente. Estas entradas fueron excluidas de la base de datos. Esto nos mejora que el agente a definir recupere únicamente contextos con alta densidad informativa, incrementando la confiabilidad de las respuestas generadas y optimizando el uso de recursos computacionales

In [ ]:
df_claria = df_sample[df_sample['Filtro'] == 'ACTIONABLE'].copy()

In [ ]:
df_claria.info()

# Agente - RAG

### **Optimizaciones previas al modelo**

In [ ]:
def evaluate_chunking_configs(texts, configs):
    results = []
    examples = {}

    for config in configs:
        size = config['chunk_size']
        overlap = config['overlap']

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=overlap,
            length_function=len
        )

        start_time = time.time()
        all_chunks = []
        for text in texts:
            all_chunks.extend(splitter.split_text(text))
        end_time = time.time()

        # Guardamos métricas
        avg_len = sum(len(c) for c in all_chunks) / len(all_chunks) if all_chunks else 0
        results.append({
            'chunk_size': size,
            'overlap': overlap,
            'total_chunks': len(all_chunks),
            'avg_chunk_length': avg_len,
            'time_sec': end_time - start_time
        })

        # Guardamos un ejemplo para ver cómo quedó
        examples[f"{size}_{overlap}"] = all_chunks[:3]

    return pd.DataFrame(results), examples

# Ahora puedes correr tu main sin importar nada externo
def main(df_input):
    texts = df_input['clean_narrative'].tolist()
    print(f"🚀 Procesando {len(texts)} quejas...")

    chunk_configs = [
        {"chunk_size": 200, "overlap": 20},
        {"chunk_size": 300, "overlap": 20},
        {"chunk_size": 400, "overlap": 20}
    ]

    results_df, examples = evaluate_chunking_configs(texts, chunk_configs)

    print("\n--- Resultados del Experimento ---")
    print(results_df)

    return results_df, examples

# Ejecutar
if 'df_sample' in globals():
    res, ex = main(df_claria)

**Observaciones**

Realizamos un experimento de optimización de segmentación (chunking) para determinar la mejor manera de fragmentar tus textos antes de enviarlos al modelo. Utilizando el RecursiveCharacterTextSplitter, el script prueba tres configuraciones diferentes de tamaño de bloque (200, 300 y 400 caracteres) manteniendo un solapamiento (overlap) de 20 caracteres para asegurar que no se pierda el contexto entre fragmentos. El objetivo principal es medir cuántos bloques se generan, cuánto tiempo tarda el proceso y cuál es la longitud promedio de cada trozo, permitiéndote elegir el equilibrio perfecto entre detalle (bloques pequeños) y contexto completo (bloques grandes) para que tu futuro buscador sea lo más preciso posible.

La configuración de 300 caracteres es la más adecuada porque logra el equilibrio óptimo entre eficiencia computacional y riqueza contextual. Genera solo 5.263 fragmentos (un poco más de la opción de 400), reduce drásticamente la carga de procesamiento y los costos operativos del agente de IA, manteniendo al mismo tiempo bloques de texto lo suficientemente extensos para capturar la narrativa completa del problema sin segmentar ideas críticas.


Procederemos a realizar una Indexación Semántica, que es el motor que permite al agente "entender" y buscar quejas por su significado profundo en lugar de simples palabras clave.
1. Fragmentaremos quejas con la configuración previa.
2. Transformaremos esas quejas, mediante el modelo all-MiniLM-L6-v2, en embeddings. Representaciones numéricas del concepto de cada reclamo.
3. Organización de los vectores en un índice FAISS, una base de datos vectorial de alta velocidad que permite al sistema identificar instantáneamente patrones comunes y "causas raíz" similares entre miles de registros, guardando todo en archivos locales para que el agente pueda consultarlos de forma eficiente sin tener que re-procesar la información cada vez.

In [ ]:
# Configuración de segmentos
CHUNK_SIZE = 300
CHUNK_OVERLAP = 20
MODEL_NAME = 'all-MiniLM-L6-v2'

def run_indexing_pipeline(df):
    # 1. CHUNKING con Metadata
    print("✂️ Generando fragmentos y metadatos...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    chunked_data = []
    for _, row in df.iterrows():
        chunks = splitter.split_text(row['clean_narrative'])
        for chunk in chunks:
            chunked_data.append({
                "complaint_id": row.get('Complaint ID', 'N/A'),
                "product": row.get('Product', 'N/A'),
                "chunk_text": chunk
            })

    df_chunks = pd.DataFrame(chunked_data)

    # 2. EMBEDDING
    print(f"🧠 Generando embeddings con {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(df_chunks['chunk_text'].tolist(), show_progress_bar=True)

    # 3. INDEXING con FAISS
    print("🗂️ Creando índice FAISS...")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype('float32'))

    # 4. GUARDAR RESULTADOS

    faiss.write_index(index, "complaints_index.faiss")
    df_chunks.to_csv("metadata_chunks.csv", index=False)

    print(f"✅ Proceso terminado. {len(df_chunks)} fragmentos indexados.")
    return index, df_chunks


index, metadata_df = run_indexing_pipeline(df_claria)

### **Modelo**

In [ ]:
# Definimos el modelo fuera de la función para que no se cargue en cada búsqueda

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def search_complaints(query, index, metadata_df, model, k=3):
    """
    Busca los fragmentos más relevantes usando el modelo ya cargado.
    """
    # 1. Convertir la pregunta en vector
    query_vector = model.encode([query]).astype('float32')

    # 2. Buscar en FAISS
    distances, indices = index.search(query_vector, k)

    # 3. Recuperar metadatos (asegúrate de que metadata_df fue creado con el pipeline fijo)
    results = metadata_df.iloc[indices[0]].copy()
    results['score'] = distances[0]

    return results

# --- PRUEBA DE BÚSQUEDA ADAPTADA ---
pregunta = "I am having issues with unauthorized charges on my credit card"
top_results = search_complaints(pregunta, index, metadata_df, embedding_model)

print("🔍 Quejas encontradas:")
print(top_results[['product', 'complaint_id', 'chunk_text']])


**Observaciones**

Utilizamos un modelo pre-entrenado all-MiniLM-L6-v2 como un motor de traducción semántica que transforma cada queja en embeddings, es decir en vectores numéricos que capturan el significado profundo del texto. Luego FAISS los organiza en una estructura de datos optimizada para búsquedas de alta velocidad. De esta manera, hemos construido un sistema que no busca palabras exactas, sino que encuentra registros similares de forma casi instantánea basándose en la cercanía geométrica de los conceptos.

In [ ]:

class RAGPipeline:
    def __init__(self, index, metadata_df, embedding_model):
        self.index = index
        self.metadata_df = metadata_df
        self.model = embedding_model

    def generate_answer(self, query):
        # 1. Usamos tu función de búsqueda
        results = search_complaints(query, self.index, self.metadata_df, self.model, k=3)

        # 2. Obtenemos los textos de las quejas (chunks)
        chunks = results['chunk_text'].tolist()

        # 3. Generamos la respuesta (Aquí podrías conectar un modelo como Gemini o GPT)
        # Por ahora, simulamos la respuesta uniendo los hallazgos
        answer = f"Based on {len(chunks)} records, the main issues are: " + " ".join(chunks[:2])

        return answer, chunks

# Inicializamos el sistema
rag = RAGPipeline(index, metadata_df, embedding_model)

**Observaciones**

En esta salida cargamos el modelo de inteligencia artificial y definimos la lógica de búsqueda. La idea es traducir la pregunta a números y rastrear en milisegundos dentro de la base de datos para encontrar las quejas que más se parecen a lo que estás consultando.

In [ ]:
# 1. Definir el modelo correctamente
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Función corregida (fíjate en el argumento 'model')
def search_complaints(query, index, metadata_df, model, k=3):
    """
    Busca fragmentos usando el objeto 'model' directamente.
    """
    # 1. Convertir la pregunta en vector usando el objeto ya cargado
    query_vector = model.encode([query]).astype('float32')

    # 2. Buscar en FAISS
    distances, indices = index.search(query_vector, k)

    # 3. Recuperar metadatos
    results = metadata_df.iloc[indices[0]].copy()
    results['score'] = distances[0]

    return results

# 3. Prueba de fuego
pregunta = "I am having issues with unauthorized charges on my credit card"
# IMPORTANTE: Aquí pasamos 'embedding_model' (el objeto)
top_results = search_complaints(pregunta, index, metadata_df, embedding_model)

print("🔍 Quejas encontradas:")
print(top_results[['product', 'chunk_text']])

**Observaciones**

Este codigo organiza todo dentro de una "caja". Define cómo el agente debe recibir una pregunta, llamar al buscador y recolecta los fragmentos de texto relevantes para prepararlos antes de darte una respuesta final.

In [ ]:
class RAGPipeline:
    def __init__(self, index, metadata_df, model):
        self.index = index
        self.metadata_df = metadata_df
        self.model = model

    def generate_answer(self, query):
        # Llama a la función que acabamos de corregir
        results = search_complaints(query, self.index, self.metadata_df, self.model, k=3)
        chunks = results['chunk_text'].tolist()

        # Respuesta simple para el loop de evaluación
        answer = "Based on complaints: " + " | ".join(chunks[:2])
        return answer, chunks

# Inicializar
rag = RAGPipeline(index, metadata_df, embedding_model)

**Observaciones**

Este codigo "enciende" la máquina. Une el cerebro, los datos y la lógica en un solo objeto listo para usar, permitiéndote que a partir de ahora solo tengas que hacerle preguntas al agente.

En definitiva el proceso previo tiene como resultado:

Si se realiza la pregunta: "¿Cuáles son los motivos principales por los que los clientes reclaman cargos no reconocidos en sus tarjetas de crédito?"

El Cerebro toma la pregunta sobre "cargos no reconocidos" y entiende que, aunque el cliente no use la palabra exacta (quizás escribió "fraude" o "me sacaron plata"), debido a su significado.

La Estructura: El RAGPipeline activa el buscador, recorre las miles de quejas que indexamos y selecciona los 3 fragmentos de texto donde la gente explica mejor ese problema específico.

Como resultado, en vez de dar una lista de 3000 quejas, devuelve los fragmentos exactos donde se menciona, por ejemplo, que los cargos aparecen después de una compra online o que son cobros duplicados.

### **Evaluación del modelo**

In [ ]:
def test_retrieval(query, k=3):
    print(f"\n🔍 BUSCANDO: '{query}'")
    print("-" * 50)

    # 1. Tu función de búsqueda en FAISS
    resultados = search_complaints(query, index, metadata_df, embedding_model, k=k)

    # 2. Mostramos los resultados de forma estética
    for i, row in resultados.iterrows():
        print(f"📍 FUENTE {i+1} | Score: {row['score']:.4f} | Categoría: {row['product']}")
        print(f"📝 Fragmento: {row['chunk_text'][:250]}...")
        print("-" * 30)

    return resultados

# Probamos con una pregunta clave
test_results = test_retrieval("problems with interest rates on my loan")

Para evaluar si el agente RAG es útil como motor de búsqueda en coincidencia de quejas anteriores miraremos:
1. Score: Un score entre 0.7-0.9 implica una relación semántica fuerte, si es mayor a 1 el agente esta "adivinando".
2. Consistencia de categoría: Si preguntas por "Credit Card" y los 3 resultados dicen "Credit Card" el agente detecta coincidencias y contexto.
3. Fragmentación: Como realizamos un corte en 300 carácteres, mide si ese corte logra captar la idea general del comentario.

###**Observaciones**

1. Consistencia en categorías. Al preguntarle sobre "interest rates", los tres fragmentos hablan exactamente de eso: pagos que no cubren intereses, tasas que suben cada tres meses y tasas "despreciables". Además el sistema fue capaz de encontrar el problema en Student Loans y en Personal Loans. Esto es muy bueno, porque significa que el motor no está "sesgado" a un solo tipo de producto bancario.

2.  Scores entre 0.80 - 0.83, hay una relación semántica fuerte. El sistema entiende el concepto aunque las palabras no sean idénticas.

3. Fragmentación: observando el resultado, el tamaño de 300 caracteres fue suficiente para capturar la queja completa.

In [ ]:
# Pregunta totalmente ajena al mundo financiero
pregunta_trampa = "What is the recipe for a chocolate cake?"

print(f"🧪 PRUEBA TRAMPA: '{pregunta_trampa}'")
print("-" * 50)


resultados_trampa = search_complaints(pregunta_trampa, index, metadata_df, embedding_model, k=3)

for i, row in resultados_trampa.iterrows():
    print(f"📍 FUENTE {i+1} | Score: {row['score']:.4f}")
    print(f"📝 Texto recuperado: {row['chunk_text'][:150]}...")
    print("-" * 30)

**Observaciones**

Vemos que de la pregunta realizada sin ninguna relación con el contexto financiero, el agente no recupera ningún fragmento.

###**Ajuste de hiperparametros**

Decidimos ver que cantidad de fragmentos optimiza el valor de score, para mejorar la relación semántica y que el modelo no adivine ni se quede sin respuesta.


In [ ]:
def evaluar_k_optimo(query, index, metadata_df, model, k_values=[1, 3, 5, 10, 20]):
    scores_medios = []

    print(f"📊 Evaluando K óptimo para: '{query}'\n")

    for k in k_values:
        # Buscamos en FAISS
        query_vector = model.encode([query]).astype('float32')
        distances, _ = index.search(query_vector, k)

        # Calculamos el promedio de las distancias (Scores)
        avg_score = np.mean(distances[0])
        scores_medios.append(avg_score)

        print(f"Para K={k:2d} -> Score promedio: {avg_score:.4f}")

    # Graficamos el "Método del Codo"
    plt.figure(figsize=(8, 4))
    plt.plot(k_values, scores_medios, marker='o', linestyle='--', color='#ff4f63')
    plt.title("Análisis de Degradación de Similitud (Búsqueda del K)")
    plt.xlabel("Valor de K (Cantidad de fragmentos)")
    plt.ylabel("Score Promedio (Menos es mejor)")
    plt.grid(True, alpha=0.3)
    plt.show()

# --- EJECUCIÓN ---
evaluar_k_optimo("interest rates and monthly payments", index, metadata_df, embedding_model)

###**Observaciones**

En el código anterior, creamos una gráfica para determinar la cantidad óptima de comentarios vecinos para el agente. Vemos que elegir un k muy bajo las respuestas van a ser muy rápidas y con menos ruido, pero nos perderíamos la información. Un k entre 10 y 20, nos aseguraríamos que no se pierda nada, pero los últimos fragmentos casi que no tiene que ver con la pregunta (información irrelevante).
El valor óptimo de k que mejora el score es 5, aumentando de 0.80-0.85 a 0.869, nos devolverá suficiente contexto y alta relevancia.

In [ ]:
import textwrap

class RAGPipeline:
    def __init__(self, index, metadata_df, model):
        self.index = index
        self.metadata_df = metadata_df
        self.model = model

    def generate_answer(self, query):
        # --- AJUSTE DE HIPERPARÁMETRO: k=5 ---
        results = search_complaints(query, self.index, self.metadata_df, self.model, k=5)
        chunks = results['chunk_text'].tolist()

        formatted_parts = ["🤖 **ANÁLISIS DE QUEJAS RECUPERADAS (k=5)**\n"]

        for i, chunk in enumerate(chunks):

            clean_chunk = chunk.strip()
            if len(clean_chunk) > 10:
                wrapped = textwrap.fill(f"{i+1}. {clean_chunk}", width=80)
                formatted_parts.append(wrapped)

        final_answer = "\n\n".join(formatted_parts)

        return final_answer, chunks


rag = RAGPipeline(index, metadata_df, embedding_model)

pregunta = "Which are the top 5 mortgages issues?"
respuesta, fuentes = rag.generate_answer(pregunta)

print(respuesta)

# Agente - OLLAMA

Para continuar con nuestro agente agregamos OLLAMA.

RAG se encarga de encontrar 5 textos próximos y te da como resultado los párrafos crudos, con errores ortográficos, en su idioma. Con la inclusión de Ollama, hacemos que nuestro agente tenga la capacidad de actuar como un analista. Puede interpretar, no se limita a repetir, detecta similitudes entre textos y devuelve un parráfo propio.


In [ ]:
# 1. Instalar la dependencia necesaria (zstd)
!apt-get update && apt-get install -y zstd

# 2. Ahora sí, correr el instalador oficial de Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Instalar la librería de Python
!pip install ollama -q

# 4. Lanzar el servidor en segundo plano
import subprocess

with open("ollama.log", "w") as f:
    subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f)

# Esperamos a que el servidor caliente motores
print("Iniciando servidor...")
time.sleep(15)

# 5. Descargar el modelo
print("Descargando modelo llama3.2:1b...")
!ollama pull llama3.2:1b

In [ ]:
class RAGPipeline:
    def __init__(self, index, metadata_df, model, threshold=1.00):
        self.index = index
        self.metadata_df = metadata_df
        self.model = model
        self.threshold = threshold # Umbral de seguridad

    def generate_answer(self, query):
        # Usamos k=5
        results = search_complaints(query, self.index, self.metadata_df, self.model, k=5)


        # Si el mejor score es mayor al umbral, detenemos el proceso para evitar inventos
        best_score = results['score'].iloc[0]
        if best_score > self.threshold:
            return (f"I'm sorry, I couldn't find complaints closely related to your query "
                    f"in my database (Score: {best_score:.2f})."), []

        chunks = results['chunk_text'].tolist()
        context = "\n".join([f"- {c}" for c in chunks])

        prompt = f"""
        SYSTEM INSTRUCTIONS:
        You are a neutral and analytical Compliance Senior Assistant.
        Your goal is to summarize customer complaints to identify service failures.

        RULES:
        1. Analyze the content of the attached complaints.
        2. Summarize the main issues
        3. Do not express personal opinions; only report user facts.

        COMPLAINT CONTEXT:
        {context}

        USER QUESTION: {query}
        """

        response = ollama.chat(
            model='llama3.2:1b',
            messages=[{'role': 'user', 'content': prompt}],
            options={
                'temperature': 0.2,
                'top_p': 0.9
            }
        )

        return response['message']['content'], chunks

rag = RAGPipeline(index, metadata_df, embedding_model, threshold=1.00)

In [ ]:
pregunta_test = "What are the main issues reported regarding interest rate calculations and payment applications in mortgages?"

respuesta, fragmentos = rag.generate_answer(pregunta_test)

print("🤖 RESPUESTA DE CLARIA:")
print("-" * 30)
print(respuesta)
print("-" * 30)
print(f"📚 Fuentes utilizadas: {len(fragmentos)} chunks.")

In [ ]:
pregunta_test = "Is there evidence of growth in complaints related to digital fraud?"

respuesta, fragmentos = rag.generate_answer(pregunta_test)

print("🤖 RESPUESTA DE CLARIA:")
print("-" * 30)
print(respuesta)
print("-" * 30)
print(f"📚 Fuentes utilizadas: {len(fragmentos)} chunks.")

In [ ]:
test_queries = {
    "Direct": "What are the problems with mortgage interest rates?",
    "Out of context": "How do I make a chocolate cake?",
    "Spanish": "¿Qué quejas hay sobre préstamos estudiantiles?"
}

print(f"{'TIPO':<15} | {'SCORE':<7} | {'ESTADO'}")
print("-" * 40)

for tipo, q in test_queries.items():
    # Simulamos lo que hace tu clase internamente
    query_vector = embedding_model.encode([q]).astype('float32')
    distances, _ = index.search(query_vector, k=1)
    score = distances[0][0]

    estado = "✅ OK" if score <= 1.00 else "❌ BLOQUEADO (Correcto)"
    if score > 1.00 and tipo != "Fuera de tema":
        estado = "⚠️ ALERTA (No encontró datos)"

    print(f"{tipo:<15} | {score:<7.4f} | {estado}")

In [ ]:
pregunta_eval = "What issues are reported regarding mortgage interest rate calculations?"

respuesta, chunks = rag.generate_answer(pregunta_eval)

print("🔍 --- PASO 1: PRECISIÓN DEL CONTEXTO (RETRIEVAL) ---")

for i, c in enumerate(chunks, 1):
    print(f"Fragmento {i}: {c[:120]}...")

print("\n🤖 --- PASO 2: FIDELIDAD Y RELEVANCIA (GENERATION) ---")

print(f"PREGUNTA ORIGINAL: {pregunta_eval}")
print(f"RESPUESTA DE CLARIA:\n{respuesta}")

###**Observaciones**

En el cuadro de la última salida vemos:

1. Utilizamos un threshold=1.00 como filtro en las respuestas, si la pregunta supera 1 el agente no encontrará respuesta. En nuestro agente vemos que al tratarse de preguntas de temas no financieros o preguntas en español, no arroja respuesta.

2. Observamos que de los 5 fragmentos, logró identificar el tema específico en los 1 y 2, y de forma más general en el 4 y 5. El motor encontró quejas de hipotecas e intereses

3. El modelo no inventó números, ni nombres de bancos, ni leyes que no estaban. Se mantuvo dentro de lo que leyó.

4.  El agente logró realizar un análisis basándose en los fragmentos seleccionados previamente y no generó ideas propias ni opiniones.

### **Testeo de distintas versiones de Ollama**

In [ ]:
# Llama 3.2 1b (rápido y eficiente)
!ollama pull llama3.2:1b

# Phi-3 (muy bueno en lógica, de Microsoft)
!ollama pull phi3:mini

!pip install ollama -q

# Descargamos los modelos adicionales
!ollama pull mistral
!ollama pull solar

In [ ]:
# Lanzamos el servidor y redirigimos la salida
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# La función correcta es sleep (espera en segundos)
time.sleep(5)

print("✅ Servidor de Ollama listo.")

In [ ]:
from IPython.display import display
import time # Ensure time is imported

def compare_multiple_queries_optimized(queries, model_list, rag_pipeline_instance):
    results_by_model = {model: [] for model in model_list}

    for i_query, query in enumerate(queries):
        print(f"\nProcessing query {i_query+1}/{len(queries)}: '{query}'")

        # Step 1: Use the RAG pipeline instance to get the retrieved chunks.
        # We call generate_answer mainly to leverage its retrieval and thresholding logic.
        # We discard the LLM-generated answer from this call and only use the chunks.
        rag_answer, retrieved_chunks = rag_pipeline_instance.generate_answer(query)

        # Check if the RAG pipeline returned an error message instead of actual chunks
        if not retrieved_chunks: # This means the threshold was exceeded, or some other retrieval issue
            context_for_llm = f"Retrieval failed for this query: {rag_answer}"
            print(f"  ⚠️ RAG retrieval skipped for this query due to threshold/error: {rag_answer}")
            # For comparison, we should still pass some context or a specific message to the LLMs.
        else:
            context_for_llm = "\n".join([f"- {c}" for c in retrieved_chunks])
            print(f"  ✅ RAG retrieval successful, {len(retrieved_chunks)} chunks retrieved.")


        for model_name in model_list:
            print(f"  🚀 PROCESANDO CON: {model_name} para la query...")
            inicio_modelo = time.time()

            prompt = f"""
SYSTEM INSTRUCTIONS:
You are a neutral and analytical Compliance Senior Assistant.
Your goal is to summarize customer complaints to identify service failures.

RULES:
1. Analyze the content of the attached complaints.
2. Summarize the main issues
3. Do not express personal opinions; only report user facts.

COMPLAINT CONTEXT:
{context_for_llm}

USER QUESTION: {query}
"""

            try:
                response = ollama.chat(
                    model=model_name,
                    messages=[{'role': 'user', 'content': prompt}],
                    keep_alive=0,
                    options={
                        "temperature": 0,
                        "num_predict": 200
                    }
                )

                content = response['message']['content'].strip()
                results_by_model[model_name].append(content)
                print(f"   ✅ Respuesta para {model_name} finalizada.")

            except Exception as e:
                results_by_model[model_name].append(f"Error: {str(e)}")
                print(f"   ❌ Error en {model_name} para esta query: {e}")

            fin_modelo = time.time()
            print(f"  ⏱️ Tiempo para {model_name}: {round(fin_modelo - inicio_modelo, 2)} seg.")

    # Construct the final report
    df = pd.DataFrame({"Complaint": queries})
    for model_name in model_list:
        # Ensure all lists have the same length as queries, filling with default if shorter
        while len(results_by_model[model_name]) < len(queries):
            results_by_model[model_name].append("N/A - Internal processing error") # Fallback for unexpected issues
        df[model_name] = results_by_model[model_name]

    return df


# --- CONFIGURACIÓN DE PRUEBA ---

quejas_reales = [
    "My credit card was cloned and they spent $2,000. The bank is not responding.",
    "The ATM was out of service for two hours on Sunday.",
    "I've been trying to cancel my account for months and they keep charging fees."
]

mis_modelos = ['llama3.2:1b', 'phi3:mini', 'mistral', 'solar']

# The 'rag' object (instance of RAGPipeline) must be initialized and available from previous cells.
df_final = compare_multiple_queries_optimized(quejas_reales, mis_modelos, rag)

df_final.rename(columns={
    'llama3.2:1b': 'Llama3.2-1B',
    'phi3:mini': 'Phi3-Mini',
    'mistral': 'Mistral',
    'solar': 'Solar',
}, inplace=True)

display(df_final)

In [ ]:
pd.set_option('display.max_colwidth', None)
df_final

**Llama 3.2 1B**:
Mantiene una estructura profesional con resúmenes, puntos clave y análisis técnico. Es el más fiel a la brevedad de la queja sin inventar demasiada "historia" de fondo.

Puede generar falsos datos como en el segundo ejemplo en dónde hace alución a contactos previos cuando el comentario no lo menciona.


**Phi-3 Mini (Microsoft)**
Tiende a fragmentar la información como si hubiera múltiples casos o IDs de queja, incluso cuando la entrada es una sola frase. Puede ser útil para categorizar grandes volúmenes, pero pierde precisión en quejas individuales.



**Mistral**
Destaca por crear listas numeradas muy detalladas que exploran las consecuencias del problema (como el impacto en el historial crediticio o el estrés financiero).

Un poco más general que Solar en acciones específicas.

**Solar**
Prefiere el formato de párrafo y se enfoca en el sentimiento general y los fallos de servicio sistémicos. Es directo, pero menos "escaneable" visualmente que Mistral o Llama.

A veces simplifica demasiado.
Menos explicación de riesgo sistémico.

# Limitaciones

1. Volumen y recursos computacionales: El dataset de más de 1.2 millones de registros requirió un trabajo significativo de muestreo y filtrado, limitando potencialmente la representatividad de algunos análisis.
2. Datos anonimizados: Las secuencias de 'X' que reemplazan datos confidenciales introducen ruido significativo en los embeddings. La solución implementada requirió un trabajo adicional antes de procesar el RAG.
3. Para simplificar el trabajo, filtramos el dataset solo para bancos. El mismo cuenta con diversas instituciones que podrían mejorar el agente.


# Conclusiones

El proyecto ClarIA demostró que es posible transformar un conjunto masivo de quejas financieras no estructuradas en insights accionables mediante la combinación inteligente de técnicas de Deep Learning: LLM con RAG. Los resultados principales se pueden resumir en estas  dimensiones:

1. La arquitectura propuesta: limpieza → embeddings semánticos → índice FAISS → modelo de lenguaje generativo — demostró ser técnicamente viable para trabajar con quejas financieras.
2. El agente va más allá de una simple búsqueda de palabras clave: al comprender el contexto semántico de las quejas, puede identificar causas raíz compartidas entre quejas con vocabulario diferente (no idiomas), detectar tendencias emergentes antes de que escalen,  distinguir entre problemas sistémicos del sector.
3. Para instituciones financieras, ClarIA puede ser una gran herramienta para la gestión proactiva de riesgos de reputación y operativos. En lugar de reaccionar a quejas individuales, el sistema permite identificar patrones estructurales que, una vez corregidos, pueden prevenir miles de quejas futuras. Esto lo logra, a su vez, eficientizando la operativa, ya que se evita el tiempo de lectura manual. Los picos temporales detectados en el análisis exploratorio son ejemplos de eventos sistémicos que un sistema en tiempo real podría alertar con días o semanas de anticipación.
4. De los agentes analizados, Llama 3.2 1B es el que proporciona más precisión en las respuestas y tiene menos alucinaciones.



# Proximos Pasos

1. Escalar el sistema a datasets más grandes.
2. Incorporar datos estructurados al agente, como puede ser: fecha del reclamo, empresa, datos del cliente, datos de los productos del cliente con el banco, etc.
3. Implementar un pipeline de actualización continua que incorpore nuevas quejas en tiempo real.
4. Desarrollar un dashboard interactivo que permita a analistas  consultar el sistema en lenguaje natural y explorar los tópicos emergentes visualmente.
5. Ampliar el agente para que tome contexto de cualquier entidad financiera.
